# 00 — Verify Setup

Run this notebook first on Kaggle to confirm:
- GPU is available and recognised
- PyTorch version is correct
- All required libraries are importable
- The model architecture is correct (shape + parameter count)
- The dataset class works end-to-end

Fix anything that fails here before running data preparation or training.

In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
from PIL import Image

print(f'PyTorch version:     {torch.__version__}')
print(f'TorchVision version: {torchvision.__version__}')
print(f'NumPy version:       {np.__version__}')
print(f'PIL version:         {Image.__version__}')

print(f'\nGPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name:      {torch.cuda.get_device_name(0)}')
    print(f'GPU memory:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Training will be very slow.')
    print('On Kaggle: Settings → Accelerator → GPU T4 x2')

In [ ]:
# Add source directory to path (adjust if running from a different location)
import sys
sys.path.append('/kaggle/input/goalz-custom-model-src')  # Kaggle utility dataset
# OR if running locally:
# sys.path.append('../')  # parent of notebooks/

from model import NatureResNet, CLASSES  # CLASSES is also exported from dataset.py
print(f'Classes ({len(CLASSES)}): {CLASSES}')

In [ ]:
# Architecture sanity check
# Run this BEFORE training — it catches shape errors immediately.

model = NatureResNet(num_classes=9)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'Expected:             ~2,800,000')

dummy = torch.randn(4, 3, 224, 224)  # batch of 4 images
out = model(dummy)
print(f'\nOutput shape: {out.shape}')  # must be torch.Size([4, 9])
assert out.shape == (4, 9), f'Expected (4, 9), got {out.shape}'
print('Shape check: PASSED')

# Verify intermediate shapes by adding print statements to forward()
# or by checking each stage individually:
from model import ResidualBlock
block = ResidualBlock(32, 64, stride=2)
x = torch.randn(4, 32, 56, 56)
out_b = block(x)
assert out_b.shape == (4, 64, 28, 28), f'Block shape wrong: {out_b.shape}'
print('Block shape check: PASSED')

In [ ]:
# Optional: verify ONNX and onnxruntime are importable
import onnx
import onnxruntime as ort
print(f'ONNX version:        {onnx.__version__}')
print(f'ONNXRuntime version: {ort.__version__}')
print('All imports successful.')